# Coronary blockage demo — upload an angiogram, mark the blockage

Runs your **trained stenosis detector** (`best.pt`) on one image and draws a **red box on each blockage** (+ confidence). If you also point it at the **vessel-seg student** (`student.pt`), it tints the vessel tree green for context.

Coronary angiogram = **arteries only** → this marks *arterial* blood blockage (stenosis). Vein/AVF blockage is a different scan + data, not this model.

Works on the same Kaggle/Colab session where you trained the detector.

In [ ]:
# 1) repo + ultralytics (skip if already set up in this session)
import os, sys, glob
REPO = '/kaggle/working/interventional-imaging-pipeline'
if not os.path.exists(REPO):
    REPO = '/content/interventional-imaging-pipeline'
    if not os.path.exists(REPO):
        !git clone https://github.com/jugalmodi0111/interventional-imaging-pipeline.git {REPO}
%cd {REPO}
sys.path.insert(0, REPO)
!pip -q install 'ultralytics>=8.2' opencv-python 2>/dev/null
print('ready')

In [ ]:
# 2) weights + pick an image
DETECTOR = glob.glob(f'{os.environ.get("HOME","")}/**/runs/**/stenosis/**/best.pt', recursive=True)
DETECTOR = DETECTOR[0] if DETECTOR else '/kaggle/working/intv-img/runs/stenosis/base/weights/best.pt'
SEG = ''                                     # optional: '.../runs/coronary/student.pt' for green vessel overlay
print('detector:', DETECTOR, '| exists:', os.path.exists(DETECTOR))

# IMAGE: point at any coronary angiogram. Default grabs an ARCADE stenosis test frame if present.
cands = glob.glob('/kaggle/input/**/stenosis/test/images/*.png', recursive=True) or \
        glob.glob('data/raw/arcade/stenosis/**/images/*.png', recursive=True)
IMAGE = cands[0] if cands else ''            # <- or set your own uploaded image path here
print('image   :', IMAGE)
assert os.path.exists(DETECTOR), 'Train the stenosis detector first (kaggle_stenosis_build.ipynb).'
assert IMAGE and os.path.exists(IMAGE), 'Set IMAGE to a coronary angiogram path (or upload one).'

In [ ]:
# 3) detect the blockage + show
from src.serve.predict_image import predict
import matplotlib.pyplot as plt, cv2
r = predict(IMAGE, DETECTOR, seg=(SEG or None), conf=0.25)
print(r['n_blockages'], 'blockage(s)')
plt.figure(figsize=(8, 8)); plt.axis('off')
plt.imshow(cv2.cvtColor(r['vis'], cv2.COLOR_BGR2RGB)); plt.show()

### Upload your own image (Colab)
```python
from google.colab import files
up = files.upload()                       # pick a .png/.jpg angiogram
IMAGE = list(up)[0]
```
On **Kaggle**, add the image as a Dataset (Add Input) and set `IMAGE = '/kaggle/input/<slug>/your.png'`, then re-run cell 3.

**Lower `conf`** (e.g. 0.10) to catch more blockages, **raise** it (0.4) for only confident ones. The current detector is an early model (mAP ~0.15) — expect misses/false positives until it's trained with Danilov + more epochs.